In [1]:
import pandas as pd
from lightgbm import LGBMRegressor

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
submission = pd.read_csv('submission_sample.csv')

In [4]:
train['time'] = pd.to_datetime(train['time'])
test['time'] = pd.to_datetime(test['time'])

In [12]:
train

,time,pv_id,appr_temp,ceiling,cloud_b,dew_point,precip_1h,pressure,real_feel_temp,real_feel_temp_shade,...,temp_a,temp_max,temp_min,wind_dir_a,wind_spd_a,coord1,coord2,type,energy,nins
0,2024-08-01 00:05:00+09:00,PV_ID_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-2.018131,-0.172021,train,NaN,0.0
1,2024-08-01 00:10:00+09:00,PV_ID_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-2.018131,-0.172021,train,NaN,0.0
2,2024-08-01 00:15:00+09:00,PV_ID_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-2.018131,-0.172021,train,0.0,0.0
3,2024-08-01 00:20:00+09:00,PV_ID_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-2.018131,-0.172021,train,NaN,0.0
4,2024-08-01 00:25:00+09:00,PV_ID_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-2.018131,-0.172021,train,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19236943,2025-07-31 23:40:00+09:00,PV_ID_209,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.971977,0.318426,train,NaN,0.0
19236944,2025-07-31 23:45:00+09:00,PV_ID_209,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.971977,0.318426,train,0.0,0.0
19236945,2025-07-31 23:50:00+09:00,PV_ID_209,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.971977,0.318426,train,NaN,0.0
19236946,2025-07-31 23:55:00+09:00,PV_ID_209,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.971977,0.318426,train,NaN,0.0


In [18]:
train["pv_id"].value_counts()


pv_id
PV_ID_0      105120
PV_ID_132    105120
PV_ID_135    105120
PV_ID_136    105120
PV_ID_137    105120
              ...  
PV_ID_90     105119
PV_ID_25     105119
PV_ID_101    105119
PV_ID_16     105119
PV_ID_40     105113
Name: count, Length: 183, dtype: int64

In [13]:
train.loc[0]

time                    2024-08-01 00:05:00+09:00
pv_id                                     PV_ID_0
appr_temp                                     NaN
ceiling                                       NaN
cloud_b                                       NaN
dew_point                                     NaN
precip_1h                                     NaN
pressure                                      NaN
real_feel_temp                                NaN
real_feel_temp_shade                          NaN
rel_hum                                       NaN
temp_b                                        NaN
uv_idx                                        NaN
vis                                           NaN
wind_chill_temp                               NaN
wind_dir_b                                    NaN
wind_gust_spd                                 NaN
wind_spd_b                                    NaN
cloud_a                                       NaN
ground_press                                  NaN


In [ ]:
features = train.columns.drop(['time', 'pv_id', 'type', 'energy', 'nins'])

dfs = []
for pv_id, df_group in train.groupby('pv_id'):
    dfs.append(df_group[features].apply(lambda x: x.fillna(method='bfill')))
weather_fillna_df = pd.concat(dfs)
train[features] = weather_fillna_df

dfs = []
for pv_id, df_group in test.groupby('pv_id'):
    dfs.append(df_group[features].apply(lambda x: x.fillna(method='bfill')))
weather_fillna_df = pd.concat(dfs)
test[features] = weather_fillna_df

In [ ]:
train = train.dropna(subset=features)

In [ ]:
lgbm = LGBMRegressor()
lgbm.fit(train[features], train['nins'])
lgbm_pred = lgbm.predict(test[features])

In [ ]:
lgbm_pred[lgbm_pred < 0] = 0

In [ ]:
submission['nins'] = lgbm_pred
submission.to_csv('result_submssion.csv')
